<a href="https://colab.research.google.com/github/Sinemvrlk/AIF360/blob/main/Assignment_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Assignment 4: Implementation Adversial Debiasing

In [29]:
# Install libraries
%pip install -q "aif360[AdversarialDebiasing]==0.6.1" "scikit-learn==1.5.2" pandas numpy

In [30]:
# Importing libraries
import numpy as np
import pandas as pd
import tensorflow as tf
import sklearn
import aif360

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score
from sklearn.neighbors import NearestNeighbors

from aif360.sklearn.datasets import (
    fetch_adult,
    fetch_german,
    fetch_compas
)
from aif360.sklearn.inprocessing import AdversarialDebiasing

tf.compat.v1.disable_eager_execution()

SEED = 42
TEST_SIZE = 0.20
EPOCHS = 30
BATCH_SIZE = 128

np.random.seed(SEED)

print("TensorFlow:", tf.__version__)
print("Scikit-learn:", sklearn.__version__)
print("AIF360:", aif360.__version__)

TensorFlow: 2.21.0
Scikit-learn: 1.5.2
AIF360: 0.6.1


In [31]:
def fairness_metrics(y, protected, X_similarity):
    y = np.asarray(y).astype(int)
    protected = np.asarray(protected).astype(int)

    # Protected groups
    p_priv = y[protected == 1].mean()
    p_unpriv = y[protected == 0].mean()

    dp_diff = p_unpriv - p_priv

    dir_ratio = (
        p_unpriv / p_priv
        if p_priv > 0 else np.nan
    )

    # Consistency using 5 nearest neighbors
    n_neighbors = min(6, len(y))

    nn = NearestNeighbors(
        n_neighbors=n_neighbors
    )
    nn.fit(X_similarity)

    indices = nn.kneighbors(
        X_similarity,
        return_distance=False
    )

    differences = []

    for i, neighbors in enumerate(indices):
        neighbors = neighbors[neighbors != i][:5]

        if len(neighbors) > 0:
            differences.append(
                abs(y[i] - y[neighbors].mean())
            )

    consistency = 1 - np.mean(differences)

    return {
        "DP_Diff": dp_diff,
        "DIR": dir_ratio,
        "Consistency": consistency
    }

In [32]:
datasets = {
    "Adult": {
        "loader": lambda: fetch_adult(),
        "protected": "sex",
        "privileged": "Male",
        "positive": ">50K"
    },
    "German": {
        "loader": lambda: fetch_german(),
        "protected": "age",
        "privileged": "aged",
        "positive": "good"
    },
    "COMPAS": {
        "loader": lambda: fetch_compas(
            binary_race=True
        ),
        "protected": "race",
        "privileged": "Caucasian",
        "positive": "Survived"
    }
}

print("Dataset configurations ready.")

Dataset configurations ready.


In [33]:
from aif360.sklearn.datasets import fetch_adult, fetch_german, fetch_compas

adult = fetch_adult()
german = fetch_german()
compas = fetch_compas(binary_race=True)

datasets = {
    "Adult": (adult, "sex", "Male", ">50K"),
    "German": (german, "age", "aged", "good"),
    "COMPAS": (compas, "race", "Caucasian", "Survived")
}

print("All three datasets loaded!")

All three datasets loaded!


In [34]:
# Run the bias mitigation experiment on all three datasets
results = []

for name, (data, attribute, privileged, positive) in datasets.items():

    X = data.X.copy()
    y = pd.Series(np.asarray(data.y))

    # Convert the labels to 0 and 1 and identify the protected groups
    y = (y == positive).astype(int)
    protected = (X.index.get_level_values(attribute) == privileged).astype(int)

    X = X.reset_index(drop = True)
    X = X.drop(columns=["sex", "race", "age"], errors = "ignore")

    # Split into training and testing data
    groups = y.astype(str) + "_" + pd.Series(protected).astype(str)

    X_train, X_test, y_train, y_test, p_train, p_test = train_test_split(
        X, y, protected, test_size = 0.2, random_state=42, stratify = groups
    )

    # Convert features into numbers
    X_train = pd.get_dummies(X_train)
    X_test = pd.get_dummies(X_test)

    X_train, X_test = X_train.align(X_test, join = "left", axis = 1, fill_value = 0)

    X_train = X_train.fillna(0)
    X_test = X_test.fillna(0)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Original fairness
    X_original = pd.get_dummies(X).reindex(
        columns = pd.get_dummies(X.iloc[pd.Series(y_train).index]).columns,
        fill_value = 0
    ).fillna(0)

    original = fairness_metrics(y, protected, X_original)

    results.append({
        "Dataset": name,
        "Stage": "Original",
        **original,
        "Accuracy": np.nan
    })

    # Defining train and test data
    X_train = pd.DataFrame(X_train, index = pd.Index(p_train, name = attribute))
    X_test = pd.DataFrame(X_test, index = pd.Index(p_test, name = attribute))

    # Baseline and mitigated model
    for debias in [False, True]:
        tf.compat.v1.reset_default_graph()

        model = AdversarialDebiasing(
            prot_attr = attribute,
            debias = debias,
            num_epochs = 30,
            batch_size = 128,
            random_state = 42
        )

        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

        metrics = fairness_metrics(predictions, p_test, X_test.to_numpy())

        results.append({
            "Dataset": name,
            "Stage": "Mitigated" if debias else "Baseline",
            **metrics,
            "Accuracy": accuracy_score(y_test, predictions)
        })

        if hasattr(model, "sess_"):
            model.sess_.close()

# Show the results for all three datasets
results_df = pd.DataFrame(results)
display(results_df.round(4))

,Dataset,Stage,DP_Diff,DIR,Consistency,Accuracy
0,Adult,Original,-0.1989,0.3635,0.8175,NaN
1,Adult,Baseline,-0.1884,0.3228,0.9341,0.8506
2,Adult,Mitigated,-0.0945,0.5815,0.9339,0.8438
3,German,Original,-0.1494,0.7948,0.6020,NaN
4,German,Baseline,-0.0858,0.8923,0.7880,0.8000
5,German,Mitigated,-0.0689,0.9085,0.7810,0.8100
6,COMPAS,Original,-0.1321,0.7830,0.5873,NaN
7,COMPAS,Baseline,-0.2069,0.6919,0.8129,0.6626
8,COMPAS,Mitigated,-0.1983,0.7118,0.8064,0.6654


##### Adversarial debiasing improved demographic parity across all three datasets. Adult showed the largest improvement, with only a small decrease in accuracy. German Credit improved in both fairness and accuracy, while COMPAS showed smaller improvements. These results support the paper, although their main experiment focused on equalized odds rather than demographic parity.